# mpostele - Wan2.1 remote dispatch server

Run every cell below, top to bottom, once per session. The last cell prints a
public ngrok URL - copy it into `WAN21_REMOTE_ENDPOINT` on your local machine
(and make sure `WAN21_API_KEY` there matches `API_KEY` in the config cell).

This has to stay running for the whole time you want remote dispatch
available. Colab has no API to start this notebook headlessly - someone has
to open it and press Run each session. See
docs-mpostele/03 Workflow/03 Video Rendering Path.md for the full picture.

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio "diffusers>=0.32" transformers accelerate safetensors

In [ ]:
# --- Configuration - fill these in before running the rest ---

NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"  # https://dashboard.ngrok.com/get-started/your-authtoken
API_KEY = "PASTE_A_RANDOM_SHARED_SECRET_HERE"          # must match WAN21_API_KEY on the local machine
WAN21_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

In [ ]:
import queue
import threading
import traceback
import uuid
from pathlib import Path

import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

OUTPUT_DIR = Path("/content/wan21_output")
OUTPUT_DIR.mkdir(exist_ok=True)

app = FastAPI()
jobs = {}  # job_id -> {"status": "queued"|"running"|"done"|"error", "path": str|None, "error": str|None}
job_queue: "queue.Queue" = queue.Queue()
_pipe = None


def get_pipeline():
    global _pipe
    if _pipe is None:
        _pipe = WanPipeline.from_pretrained(WAN21_MODEL_ID, torch_dtype=torch.bfloat16).to("cuda")
    return _pipe


class GenerateRequest(BaseModel):
    positive: str
    negative: str = ""
    num_frames: int = 33
    width: int = 480
    height: int = 832


def check_auth(x_api_key: str = Header(None)):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="bad or missing x-api-key")


def _run_job(job_id: str, req: GenerateRequest) -> None:
    try:
        jobs[job_id]["status"] = "running"
        pipe = get_pipeline()
        frames = pipe(
            prompt=req.positive,
            negative_prompt=req.negative,
            num_frames=req.num_frames,
            width=req.width,
            height=req.height,
        ).frames[0]
        out_path = OUTPUT_DIR / f"{job_id}.mp4"
        export_to_video(frames, str(out_path), fps=16)
        jobs[job_id]["path"] = str(out_path)
        jobs[job_id]["status"] = "done"
    except Exception:
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = traceback.format_exc()


def _worker() -> None:
    # One job at a time - a single Colab GPU can't safely run two Wan2.1
    # generations concurrently, and this matches the Sequential Execution
    # Contract the rest of the pipeline follows.
    while True:
        job_id, req = job_queue.get()
        _run_job(job_id, req)


threading.Thread(target=_worker, daemon=True).start()


@app.post("/generate")
def generate(req: GenerateRequest, x_api_key: str = Header(None)):
    check_auth(x_api_key)
    job_id = uuid.uuid4().hex[:12]
    jobs[job_id] = {"status": "queued", "path": None, "error": None}
    job_queue.put((job_id, req))
    return {"job_id": job_id}


@app.get("/status/{job_id}")
def status(job_id: str, x_api_key: str = Header(None)):
    check_auth(x_api_key)
    job = jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="unknown job_id")
    return {"status": job["status"], "error": job["error"]}


@app.get("/result/{job_id}")
def result(job_id: str, x_api_key: str = Header(None)):
    check_auth(x_api_key)
    job = jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="unknown job_id")
    if job["status"] != "done":
        raise HTTPException(status_code=409, detail=f"job is {job['status']}, not done")
    return FileResponse(job["path"], media_type="video/mp4")

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"WAN21_REMOTE_ENDPOINT = {public_url}")
print("Copy the URL above into WAN21_REMOTE_ENDPOINT on your local machine, then leave this cell running.")

uvicorn.run(app, host="0.0.0.0", port=8000)